In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.image as mpimg

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42

In [4]:
df = pd.read_csv('voice.csv')
print('Ukuran data  :', df.shape)
print('Missing value:', df.isna().sum().sum())
print('\nDistribusi label:')
print(df['label'].value_counts())
df.head()

Ukuran data  : (3168, 21)
Missing value: 0

Distribusi label:
label
male      1584
female    1584
Name: count, dtype: int64


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [5]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
print('Mapping label:', dict(zip(le.classes_, le.transform(le.classes_))))

X = df.drop(columns='label')
y = df['label']
print('X:', X.shape, '| y:', y.shape)

Mapping label: {'female': np.int64(0), 'male': np.int64(1)}
X: (3168, 20) | y: (3168,)


In [6]:
splits = {'70:30': 0.30, '80:20': 0.20}
kernels = ['linear', 'poly', 'rbf']

rows = []
for split_name, test_size in splits.items():
    # 1) split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y)

    # 2) standardisasi (fit hanya di train)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    for kernel in kernels:
        # 3) latih model
        model = SVC(kernel=kernel)
        model.fit(X_train_s, y_train)

        # 4) evaluasi
        acc_train = accuracy_score(y_train, model.predict(X_train_s))
        acc_test = accuracy_score(y_test, model.predict(X_test_s))
        rows.append((split_name, kernel, len(X_train), len(X_test), acc_train, acc_test))

result = pd.DataFrame(rows, columns=['Split', 'Kernel', 'n_train', 'n_test', 'Accuracy Train', 'Accuracy Test'])
result.round(4)

,Split,Kernel,n_train,n_test,Accuracy Train,Accuracy Test
0,70:30,linear,2217,951,0.9720,0.9790
1,70:30,poly,2217,951,0.9666,0.9600
2,70:30,rbf,2217,951,0.9838,0.9832
3,80:20,linear,2534,634,0.9767,0.9748
4,80:20,poly,2534,634,0.9680,0.9558
5,80:20,rbf,2534,634,0.9838,0.9826


In [7]:
table_acc = result.pivot(index='Kernel', columns='Split', values='Accuracy Test').loc[kernels]
table_acc_pct = (table_acc * 100).round(2).astype(str) + ' %'
print('Accuracy (data test):')
display(table_acc.round(4))
display(table_acc_pct)

Accuracy (data test):


Split,70:30,80:20
Kernel,,
linear,0.9790,0.9748
poly,0.9600,0.9558
rbf,0.9832,0.9826


Split,70:30,80:20
Kernel,,
linear,97.9 %,97.48 %
poly,96.0 %,95.58 %
rbf,98.32 %,98.26 %


## Analysis of Question 1

RBF performed best (≈98.3%), followed by Linear (≈97.5–97.9%) and Polynomial (≈95.6–96.0%). RBF works best because it can detect complex, non-linear patterns in the 20 acoustic features. The split ratio had little effect, with accuracy differences below 0.5%. No significant overfitting was found because training and testing accuracy were very similar.

In [8]:
train_dir = "images/training/"
test_dir = "images/test/"

def load_dataset(img_dir):
    p = Path(img_dir)
    img_list = []
    for d in p.glob('*'):              
        label = d.name
        for file in d.glob('*.jpg'):
            img = mpimg.imread(file)
            if img is not None:
                img_list.append((img, label))
    return img_list

all_img = load_dataset(train_dir) + load_dataset(test_dir)
print('Total gambar:', len(all_img))
print('Day  :', sum(1 for _, l in all_img if l == 'day'))
print('Night:', sum(1 for _, l in all_img if l == 'night'))

Total gambar: 400
Day  : 200
Night: 200


In [9]:
def standarized_input(image):
    return cv2.resize(image, (1100, 600))

def label_encoder(label):
    return 1 if label == 'day' else 0

def preprocess(img_list):
    out = []
    for img, label in img_list:
        out.append((standarized_input(img), label_encoder(label)))
    return out

std_img_list = preprocess(all_img)
print('Contoh shape gambar:', std_img_list[0][0].shape, '(tinggi, lebar, channel)')

Contoh shape gambar: (600, 1100, 3) (tinggi, lebar, channel)


In [ ]:
def histogram_feature(image, bins=256):
    feats = []
    for ch in range(3): 
        hist, _ = np.histogram(image[:, :, ch], bins=bins, range=(0, 256))
        feats.append(hist / image[:, :, ch].size)   # normalisasi
    return np.concatenate(feats)

def extract_histogram_features(img_list):
    X = np.array([histogram_feature(img) for img, _ in img_list])
    y = np.array([label for _, label in img_list])
    return X, y

X_img, y_img = extract_histogram_features(std_img_list)
print('Matriks fitur:', X_img.shape)

Matriks fitur: (400, 768)


In [11]:
Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    X_img, y_img, test_size=0.20, random_state=RANDOM_STATE, stratify=y_img)
print('Train:', Xi_train.shape, '| Test:', Xi_test.shape)

Train: (320, 768) | Test: (80, 768)


In [12]:
base = make_pipeline(StandardScaler(), SVC(kernel='rbf'))
base.fit(Xi_train, yi_train)

acc_base_train = accuracy_score(yi_train, base.predict(Xi_train))
acc_base_test = accuracy_score(yi_test, base.predict(Xi_test))
print(f'[Baseline RBF] Accuracy train: {acc_base_train:.4f}')
print(f'[Baseline RBF] Accuracy test : {acc_base_test:.4f}')

[Baseline RBF] Accuracy train: 0.9938
[Baseline RBF] Accuracy test : 0.9875


In [13]:
param_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 0.0001, 0.001, 0.01],
}

grid = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel='rbf')),
                    param_grid, cv=5)
grid.fit(Xi_train, yi_train)

print('Parameter terbaik :', grid.best_params_)
print(f'Akurasi CV terbaik: {grid.best_score_:.4f}')

Parameter terbaik : {'svc__C': 1, 'svc__gamma': 'scale'}
Akurasi CV terbaik: 0.9906


In [14]:
best = grid.best_estimator_
y_pred = best.predict(Xi_test)

acc_tuned_train = accuracy_score(yi_train, best.predict(Xi_train))
acc_tuned_test = accuracy_score(yi_test, y_pred)
print(f'[Tuned RBF] Accuracy train: {acc_tuned_train:.4f}')
print(f'[Tuned RBF] Accuracy test : {acc_tuned_test:.4f}\n')

print(classification_report(yi_test, y_pred, target_names=['night', 'day']))
print('Confusion matrix (baris=aktual, kolom=prediksi; urutan night, day):')
print(confusion_matrix(yi_test, y_pred))

[Tuned RBF] Accuracy train: 0.9938
[Tuned RBF] Accuracy test : 0.9875

              precision    recall  f1-score   support

       night       1.00      0.97      0.99        40
         day       0.98      1.00      0.99        40

    accuracy                           0.99        80
   macro avg       0.99      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80

Confusion matrix (baris=aktual, kolom=prediksi; urutan night, day):
[[39  1]
 [ 0 40]]


In [15]:
summary = pd.DataFrame({
    'Model': ['SVM RBF default', f"SVM RBF tuned {grid.best_params_}"],
    'Split': ['80:20', '80:20'],
    'Accuracy Train': [acc_base_train, acc_tuned_train],
    'Accuracy Test': [acc_base_test, acc_tuned_test],
})
display(summary.round(4))

# Semua kombinasi C & gamma (untuk lampiran laporan)
cv_table = pd.DataFrame(grid.cv_results_)[['param_svc__C', 'param_svc__gamma', 'mean_test_score']]
cv_table.columns = ['C', 'gamma', 'CV accuracy']
cv_table.sort_values('CV accuracy', ascending=False).round(4).reset_index(drop=True)


,Model,Split,Accuracy Train,Accuracy Test
0,SVM RBF default,80:20,0.9938,0.9875
1,"SVM RBF tuned {'svc__C': 1, 'svc__gamma': 'sca...",80:20,0.9938,0.9875


,C,gamma,CV accuracy
0,1.0,0.001,0.9906
1,1.0,scale,0.9906
2,10.0,0.001,0.9906
3,100.0,0.001,0.9906
4,100.0,scale,0.9906
5,100.0,0.0001,0.9906
6,10.0,scale,0.9906
7,1.0,0.0001,0.9875
8,10.0,0.0001,0.9875
9,0.1,scale,0.9750


## Analysis of Question 2

An SVM with an RBF kernel and RGB histogram features can classify day and night images with 98.75% test accuracy. The default parameters were already the best choice. Tuning did not improve the result because the default parameters were already the best.